# Week 4: Transfer Learning, BERT (Homework)

## Question Search Engine

Embeddings are a good source of information for solving various tasks. For example, we can classify texts or find similar documents using their representations. We already know about word2vec, GloVe and fasttext, but they don't use context information from given text (only from contexts of source data).

For today we will use full power of context-aware embeddings to find text duplicates!

__Warning:__ this task assumes you have seen `seminar.ipynb`!

## Rules

This notebook is checked automatically by **ThetaGrader (TG)**.

- Keep every `### Task N` heading intact -- TG uses it as a marker to locate tasks. Do not remove, rename, renumber, or duplicate these headings.
- Write your solution in the cell(s) reserved for it, directly below the corresponding task statement. Do not move a solution to a different place in the notebook or merge several tasks' solutions together.
- If a task asks for a written conclusion, report, or answer to a theoretical question, put it in a markdown cell directly under that task, clearly separated from code.
- Before submitting your final solution, make sure the `Status` field in the Bot-check cell below is set to `final`.

## AI usage disclosure

If you used AI assistants (e.g. ChatGPT, Claude, GitHub Copilot) while working on this homework, briefly describe here which tool(s) you used, for which parts of the assignment, and how.

Leave this cell as "no AI assistance used" if you did not use any.

In [1]:
%pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets

Note: you may need to restart the kernel to use updated packages.


### Data Preparation

In [2]:
qqp = datasets.load_dataset("SetFit/qqp")
print("\n")
print("Sample[0]:", qqp["train"][0])
print("Sample[3]:", qqp["train"][3])

Repo card metadata block was not found. Setting CardData to empty.




Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [3]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/320 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  433MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [4]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [5]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [6]:
print(repr(qqp_preprocessed["train"][0]["input_ids"])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [11]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=128, shuffle=False, collate_fn=transformers.default_data_collator,
    num_workers = 2
)

In [14]:

device = 'cuda'

In [16]:
for batch in val_loader:
    break  # here be your training code
print("Sample batch:", batch)

with torch.no_grad():
    predicted = model(
        input_ids=batch["input_ids"].to(device),
        attention_mask=batch["attention_mask"].to(device),
        token_type_ids=batch["token_type_ids"].to(device),
    )

print("\nPrediction (probs):", torch.softmax(predicted.logits, dim=1).cpu().numpy())

Sample batch: {'labels': tensor([0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1,
        1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
        0, 0, 0, 1, 0, 1, 0, 1]), 'idx': tensor([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
         14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,
         28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,
         42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,
         56,  57,  58,  59,  60,  61,  62,  63,  64,  65,  66,  67,  68,  69,
         70,  71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,
         84,  85,  86,  87,  88,  89,  90,  91,  92,  93,  94,  95

### Task 1 (1 point):

- Measure the validation accuracy of your model. Doing so naively may take several hours. Please make sure you use the following optimizations:
  - Run the model on GPU with no_grad
  - Using batch size larger than 1
  - Use optimize data loader with num_workers > 1
  - (Optional) Use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)

In [17]:
from tqdm.auto import tqdm

@torch.inference_mode()
def compute_accuracy(model, loader, device = 'cpu', dtype = torch.float16):

    model.eval()
    model = model.to(device)

    pos = 0
    total = 0

    for batch in tqdm(val_loader):
        with torch.autocast(device_type=device, dtype=dtype):
            predicted = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                token_type_ids=batch["token_type_ids"].to(device),
            )

        probs = torch.argmax(predicted.logits, dim=1).detach()
        labels = batch['labels'].to(device)

        pos += torch.sum(probs == labels).item()
        total += probs.shape[0]

    return pos/total

accuracy = compute_accuracy(model, val_loader, device = 'cuda', dtype = torch.bfloat16)

  0%|          | 0/316 [00:00<?, ?it/s]

In [18]:
assert 0.9 < accuracy < 0.91

In [19]:

accuracy


0.908459065050705

### Training (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

### Task 2 (4 points):
- Choose Option A or Option B (only one will be graded)
- Follow all the instructions and restrictions

In [5]:
model_name = "microsoft/deberta-v3-base"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name, fast = True)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name, torch_dtype = torch.float32)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
W1106 20:40:14.165000 51571 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
/opt/homebrew/anaconda3/envs/nlp/lib/python3.10/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.den

In [6]:
def parameter_counts(model):
    return {
        "total": sum(parameter.numel() for parameter in model.parameters()),
        "trainable": sum(
            parameter.numel() for parameter in model.parameters() if parameter.requires_grad
        ),
    }

# for parameter in model.base_model.parameters():
#     parameter.requires_grad_(False)

model.train()
# model.base_model.eval()

print("Model:", parameter_counts(model))
print("Encoder:", parameter_counts(model.base_model))
print("Head:", parameter_counts(model.classifier))
print("Pooler:", parameter_counts(model.pooler))

Model: {'total': 184423682, 'trainable': 184423682}
Encoder: {'total': 183831552, 'trainable': 183831552}
Head: {'total': 1538, 'trainable': 1538}
Pooler: {'total': 590592, 'trainable': 590592}


In [7]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

qqp_preprocessed = qqp.map(preprocess_function, batched=True, load_from_cache_file=False)

BS = 32

train_set = qqp_preprocessed["train"]
val_set = qqp_preprocessed["validation"]

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [37]:

# from google.colab import userdata
# HF_TOKEN = userdata.get('HF_TOKEN')
# from huggingface_hub import login
# login()

In [40]:
from transformers import Trainer, TrainingArguments
import evaluate
import numpy as np
import scipy.special

training_args = TrainingArguments(

    output_dir="debertav3-qqp-finetuned",
    push_to_hub=True,
    hub_model_id="cybberowl/debertav3-qqp-finetuned",
    hub_strategy="checkpoint",
    hub_private_repo=True,
    hub_token = HF_TOKEN,

    per_device_train_batch_size = BS,
    num_train_epochs = 3,
    learning_rate = 2e-5,
    warmup_steps = 100,
    weight_decay = 0.0,
    gradient_accumulation_steps = 1,
    average_tokens_across_devices = True,
    max_grad_norm = 1.0,
    label_smoothing_factor = 0.0,
    bf16 = True,
    bf16_full_eval=True,
    gradient_checkpointing = False,
    torch_compile = False,
    per_device_eval_batch_size = BS,
    save_only_model = True,
    eval_on_start=True,
    save_steps = 3000,
    eval_steps = 3000,
    eval_strategy="steps",
    save_strategy="steps",
    save_total_limit=3,
    greater_is_better = True,
    use_cpu = False)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
roc_auc_metric = evaluate.load("roc_auc")

def compute_metrics(eval_pred):
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    probs = scipy.special.softmax(logits, axis=1)
    preds = probs.argmax(axis = 1)
    return accuracy_metric.compute(predictions=preds, references=labels) |\
            f1_metric.compute(predictions=preds, references=labels) |\
            roc_auc_metric.compute(prediction_scores=probs[:,1], references=labels)

trainer = Trainer(model = model,
                  args = training_args,
                  data_collator = transformers.default_data_collator,
                  train_dataset = train_set,
                  eval_dataset = val_set,
                  compute_metrics = compute_metrics)

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1,Roc Auc
0,No log,0.702126,0.368167,0.538190,0.437709


Step,Training Loss,Validation Loss,Accuracy,F1,Roc Auc
0,No log,0.702126,0.368167,0.538190,0.437709
3000,0.258863,0.249475,0.894212,0.861321,0.959374
6000,0.237777,0.245190,0.903314,0.869312,0.965365
9000,0.229295,0.218676,0.911477,0.880217,0.969618
12000,0.176755,0.225501,0.913851,0.883531,0.971031


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
# from huggingface_hub import login
# login()

In [144]:
model_name = "cybberowl/debertav3-qqp-finetuned"
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name, torch_dtype = torch.float16)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

### Finding Duplicates (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

### Task 3 (1 point):
- Implement function for finding duplicates
- Test it on several examples (at least 5)
- Check suggested duplicates and make a conclusion about model correctness

In [51]:
all_questions = qqp['train'].select_columns(['text1','text2']).map(
    lambda batch: {'text': batch['text1'] + batch['text2']}, 
    remove_columns = ['text1','text2'], batched = True)


Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

In [216]:
from tqdm.auto import tqdm

@torch.inference_mode()
def score_query(text_query, model, questions, batch_size, device = 'cpu', dtype = torch.float32, verbose = True):

    model.eval()
    model.to(device)
    probs = []
    
    for batch in tqdm(questions.batch(batch_size = batch_size), disable = not verbose, leave = False):
    
        encoded_batch = tokenizer(
            batch["text"],
            [text_query]*len(batch["text"]),
            padding="max_length",
            max_length=MAX_LENGTH,
            truncation=True,
            return_tensors='pt'
        )
        
        with torch.autocast(device_type=device, dtype=dtype):
            predicted = model(
                input_ids=encoded_batch["input_ids"].to(device),
                attention_mask=encoded_batch["attention_mask"].to(device),
                token_type_ids=encoded_batch["token_type_ids"].to(device),
            )

        p = torch.softmax(predicted.logits.float(), dim=1).cpu().numpy()[:,1]

        probs.append(p)      

    return np.concat(probs)

In [217]:
def get_duplicates(questions, scores, k):

    idx = scores.argsort()[::-1][:k]

    for i in idx:
        yield scores[i], questions['text'][i]

def print_duplicates(gen):

    for score, text in gen:
        print(f"{score:.3f} - {text}")

In [215]:
queries = [
    "How can I become healthier?",
    "How to reduce my sexual energy?",
    "Which mobile phone should I buy?",
    "How to become extremely rich?",
    "How often do you dream about Roman Empire?"
]

In [220]:
%%time
for query in queries:
    probs = score_query(text_query=query, 
                        model=model, 
                        questions=all_questions, 
                        batch_size=BS,
                        device = 'mps',
                        dtype = torch.bfloat16)

    k = 10

    print('query:', query)
    print_duplicates(get_duplicates(all_questions, probs, k))
    print('-'*100)

  0%|          | 0/22741 [00:00<?, ?it/s]

query: How can I become healthier?
0.990 - *1-(800) 439–2178* BITDEFENDER Antivirus Tech Support Number, BITDEFENDER Antivirus Support Phone Number?
0.990 - *1-(800) 439–2178* BITDEFENDER Antivirus Tech Support Number, BITDEFENDER Antivirus Support Phone Number?
0.988 - Are there people who had successful long distance relationships? Can you tell me about your successful experience with long distance relationship?
0.988 - Are there people who had successful long distance relationships? Can you tell me about your successful experience with long distance relationship?
0.988 - Are there people who had successful long distance relationships? Can you tell me about your successful experience with long distance relationship?
0.988 - Are there people who had successful long distance relationships? Can you tell me about your successful experience with long distance relationship?
0.988 - Are there people who had successful long distance relationships? Can you tell me about your successful experi

  0%|          | 0/22741 [00:00<?, ?it/s]

query: How to reduce my sexual energy?
0.983 - How do I decrease my sexual energy?
0.969 - How do I reduce sexual energy?
0.968 - How do I reduce sex drive?
0.968 - How do I reduce sex drive?
0.968 - How do I reduce sex drive?
0.968 - How do I reduce sex drive?
0.966 - *1-(800) 439–2178* BITDEFENDER Antivirus Tech Support Number, BITDEFENDER Antivirus Support Phone Number?
0.966 - *1-(800) 439–2178* BITDEFENDER Antivirus Tech Support Number, BITDEFENDER Antivirus Support Phone Number?
0.954 - How do you reduce your sex drive?
0.954 - How do you reduce your sex drive?
----------------------------------------------------------------------------------------------------


  0%|          | 0/22741 [00:00<?, ?it/s]

query: Which mobile phone should I buy?
0.987 - What are some good phones to buy?
0.986 - *1-(800) 439–2178* BITDEFENDER Antivirus Tech Support Number, BITDEFENDER Antivirus Support Phone Number?
0.986 - *1-(800) 439–2178* BITDEFENDER Antivirus Tech Support Number, BITDEFENDER Antivirus Support Phone Number?
0.980 - Which is the best smartphone I can buy?
0.980 - Which is the best smartphone I can buy?
0.977 - ( (#.! .#)) Linksys Router Tech Support Number 1 8.0.0:2.5.1:4.9.1.9 : Linksys Router Support Phone Number?
0.977 - ( (#.! .#)) Linksys Router Tech Support Number 1 8.0.0:2.5.1:4.9.1.9 : Linksys Router Support Phone Number?
0.977 - ( (#.! .#)) Linksys Router Tech Support Number 1 8.0.0:2.5.1:4.9.1.9 : Linksys Router Support Phone Number?
0.975 - Are there people who had successful long distance relationships? Can you tell me about your successful experience with long distance relationship?
0.975 - Are there people who had successful long distance relationships? Can you tell me ab

  0%|          | 0/22741 [00:00<?, ?it/s]

query: How to become extremely rich?
0.995 - How can one get insanely rich?
0.995 - How can one get insanely rich?
0.995 - How can one get insanely rich?
0.995 - How can one get insanely rich?
0.995 - How can one get insanely rich?
0.995 - How can one get insanely rich?
0.995 - How can I get very rich?
0.995 - How can I get very rich?
0.995 - How can I get very rich?
0.995 - How can I get very rich?
----------------------------------------------------------------------------------------------------


  0%|          | 0/22741 [00:00<?, ?it/s]

query: How often do you dream about Roman Empire?
0.974 - *1-(800) 439–2178* BITDEFENDER Antivirus Tech Support Number, BITDEFENDER Antivirus Support Phone Number?
0.974 - *1-(800) 439–2178* BITDEFENDER Antivirus Tech Support Number, BITDEFENDER Antivirus Support Phone Number?
0.906 - What is height og SSC CGL preparation what is height of SSC CGL preparation?
0.865 - ( (#.! .#)) Linksys Router Tech Support Number 1 8.0.0:2.5.1:4.9.1.9 : Linksys Router Support Phone Number?
0.865 - ( (#.! .#)) Linksys Router Tech Support Number 1 8.0.0:2.5.1:4.9.1.9 : Linksys Router Support Phone Number?
0.865 - ( (#.! .#)) Linksys Router Tech Support Number 1 8.0.0:2.5.1:4.9.1.9 : Linksys Router Support Phone Number?
0.859 - What is rightist ideology? What is rightist ideology?
0.851 - Are there any conspiracy theories that are probably true? Any conspiracy theories that turned out to be true?
0.851 - Are there any conspiracy theories that are probably true? Any conspiracy theories that turned out to 


**Вывод**:

1. Наивное решение работает очень долго (2 часа на запрос на mps Air M4)
2. Иногда дает похожие ответы (среди которых есть дубликаты), но часто обращает много внимания на фразы со странными символами, из-за чего могут потеряться реальные дубликаты (например на вопрос "How can I become healthier?" я находил похожие вопросы про снижение веса при помощи модели в первых 10к вопросах, но на всей выборке их уже не видно)


### Bonus: Finding Duplicates Faster (0.5 point)

Try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

### Task 4 (Bonus, 0.5 point):
- Speed up your implementation from "Finding Duplicates" part
- Capture both old and new implementation work time
- Describe your approach

Я построил tf-idf вектора из sklearn. Взял 1-2 нграммы, убрал частые слова с document frequency > 0.1, и нормировал вектора чтобы считать cosine similarity через dot product. Я беру все вопросы из train которые имеют близость >0 по tf idf и после этого снова прогоняю в цикле модель. Она даже работает немного лучше чем наивный подход, так как вопрос "*1-(800) 439–2178* BITDEFENDER Antivirus Tech Support Number, BITDEFENDER Antivirus Support Phone Number?" игнорируется еще на этапе tf-idf. Работает достаточно быстро, так как после фильтрации по tf-idf остается мало фраз

In [226]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.1,
    norm = 'l2',
    stop_words='english',
    dtype=np.float32
)



In [227]:
%%time
X = vectorizer.fit_transform(all_questions['text'])

CPU times: user 5.72 s, sys: 138 ms, total: 5.86 s
Wall time: 5.93 s


In [343]:
def pre_rank(vectorizer, X, query):

    q = vectorizer.transform([query])
    scores = (q @ X.T).toarray().ravel()
    sorting_index = np.argsort(scores)[::-1]
    sorted_scores = scores[sorting_index]
    
    idx = np.nonzero(sorted_scores)

    return sorting_index[idx], sorted_scores[idx]

In [342]:
all_questions[376257]

{'text': "Which is correct, 'healthier' or 'more healthy'?"}

In [341]:
%%time

pre_rank(vectorizer, X, queries[0])

CPU times: user 53.1 ms, sys: 9.96 ms, total: 63.1 ms
Wall time: 61.4 ms


(array([376257, 105361, 182915, 631835, 711810, 469256, 408996, 633742,
          8419, 255880,  50875, 430202, 377246, 396731, 254880,  62116,
         93086, 409996, 358559,  13895, 115843, 247795, 714036, 507232,
        114843, 120629,  12895,  63116, 360142, 454225, 121629, 455225,
        221449, 218913, 704931, 264513, 680428, 667364, 715036, 177490,
        385853, 359559, 558278, 677850, 638332,   9419, 489235, 334596,
        615452, 654412, 265513, 104361, 422862, 488235, 335596,  26973,
        427473, 725323, 655412, 107580, 406002,  57903, 183421,   5421,
        676850, 547686, 303971, 662925, 389316, 485451, 372544,  78590,
        246795, 137057,  16968, 220449, 398786, 176493, 335098, 373544,
        639332, 710810,  92086, 192021, 550222, 136057,  70012]),
 array([0.69500774, 0.5683748 , 0.531091  , 0.531091  , 0.5301285 ,
        0.50279075, 0.4895961 , 0.4817227 , 0.461783  , 0.44521314,
        0.43904182, 0.43904182, 0.42310905, 0.40842962, 0.40730774,
        0.

In [344]:
%%time
k = 10

datasets.disable_progress_bars()

for query in queries:
    
    print('query:', query)
    prerank_idx, _ = pre_rank(vectorizer, X, query)
    print(f'selected {len(prerank_idx)} questions')
    
    
    probs = score_query(text_query=query, 
                        model=model, 
                        questions=all_questions.select(prerank_idx), 
                        batch_size=BS,
                        device = 'mps',
                        dtype = torch.bfloat16)


    
    print_duplicates(get_duplicates(all_questions.select(prerank_idx), probs, k))
    print('-'*100)

query: How can I become healthier?
selected 87 questions


  0%|          | 0/3 [00:00<?, ?it/s]

0.016 - Is it healthier to drink water after a meal or not to drink water after a meal?
0.002 - Do doctors really want people to live healthier?
0.001 - Is eating chicken healthier than Eating mutton?
0.001 - Is organic food really healthier? Is it more nutritious?
0.001 - Is organic food really healthier? Is it more nutritious?
0.001 - Is it healthier to eat fruit or drink fruit?
0.001 - Is it healthier to eat fruit or drink fruit?
0.001 - What is the difference between red meat and white meat? Which one is healthier?
0.001 - What is the difference between red meat and white meat? Which one is healthier?
0.001 - What are some ways to make macaroni and cheese healthier?
----------------------------------------------------------------------------------------------------
query: How to reduce my sexual energy?
selected 4991 questions


  0%|          | 0/156 [00:00<?, ?it/s]

0.983 - How do I decrease my sexual energy?
0.969 - How do I reduce sexual energy?
0.968 - How do I reduce sex drive?
0.968 - How do I reduce sex drive?
0.968 - How do I reduce sex drive?
0.968 - How do I reduce sex drive?
0.954 - How do you reduce your sex drive?
0.954 - How do you reduce your sex drive?
0.954 - How do you reduce your sex drive?
0.954 - How do you reduce your sex drive?
----------------------------------------------------------------------------------------------------
query: Which mobile phone should I buy?
selected 14330 questions


  0%|          | 0/448 [00:00<?, ?it/s]

0.987 - What are some good phones to buy?
0.986 - *1-(800) 439–2178* BITDEFENDER Antivirus Tech Support Number, BITDEFENDER Antivirus Support Phone Number?
0.986 - *1-(800) 439–2178* BITDEFENDER Antivirus Tech Support Number, BITDEFENDER Antivirus Support Phone Number?
0.980 - Which is the best smartphone I can buy?
0.980 - Which is the best smartphone I can buy?
0.977 - ( (#.! .#)) Linksys Router Tech Support Number 1 8.0.0:2.5.1:4.9.1.9 : Linksys Router Support Phone Number?
0.977 - ( (#.! .#)) Linksys Router Tech Support Number 1 8.0.0:2.5.1:4.9.1.9 : Linksys Router Support Phone Number?
0.977 - ( (#.! .#)) Linksys Router Tech Support Number 1 8.0.0:2.5.1:4.9.1.9 : Linksys Router Support Phone Number?
0.973 - What is the best phone to buy?
0.970 - Which phone do I buy?
----------------------------------------------------------------------------------------------------
query: How to become extremely rich?
selected 1239 questions


  0%|          | 0/39 [00:00<?, ?it/s]

0.995 - How can one get insanely rich?
0.995 - How can one get insanely rich?
0.995 - How can one get insanely rich?
0.995 - How can one get insanely rich?
0.995 - How can one get insanely rich?
0.995 - How can one get insanely rich?
0.995 - How can I get very rich?
0.995 - How can I get very rich?
0.995 - How can I get very rich?
0.995 - How can I get very rich?
----------------------------------------------------------------------------------------------------
query: How often do you dream about Roman Empire?
selected 1703 questions


  0%|          | 0/54 [00:00<?, ?it/s]

0.016 - If I wanted to learn about the Roman Empire,what would be the best books to read?
0.005 - What are some of the rarely mentioned facts about the Roman Empire?
0.005 - What are some of the rarely mentioned facts about the Roman Empire?
0.004 - What are some of the rarely known facts about the Roman Empire?
0.004 - What are some of the rarely known facts about the Roman Empire?
0.004 - What are some of the rarely known facts about the Roman Empire?
0.003 - What is the best book about the Roman Empire?
0.003 - What are some of the rarely mentioned facts of the Roman empire?
0.003 - What are some rarely known facts of the roman empire?
0.003 - What are some rarely known facts of the roman empire?
----------------------------------------------------------------------------------------------------
CPU times: user 7.62 s, sys: 2.6 s, total: 10.2 s
Wall time: 4min


### Bonus: Finding Duplicates in Old-Fashioned way (1.5 points)

In this bonus task you are supposed to use pretrained embeddings (word2vec, GloVe or fasttext) for solving the duplicates problem.

### Task 5 (Bonus, 1.5 points):
- Solve Finding Duplicates problem using mentioned embeddings
- Compare old-fashioned solution to previous ones (quality, speed, etc.)
- Make a small report (up to 5 steps, results and conclusions) on work done in this part

Я загрузил вектора fast-text и считаю эмбеддинг предложения как взвешенный по idf средний вектор всех слов. Ушло чуть меньше минуты чтобы сделать расчет эмбеддингов всех вопросов, генерация ответов получается мгновенная. Хотя такой подход не дает идеально дублирующиеся фразы, они все равно похожи по смыслу и при этом там нет явно лишнего спама, как было с deberta.

In [486]:
# import gensim.downloader as api

# wv = api.load("fasttext-wiki-news-subwords-300")

In [423]:
from nltk.tokenize import WordPunctTokenizer

nltk_tok = WordPunctTokenizer()

def compute_embed(text, wv, tokenizer, idf_scores):
    words = text.lower()
    tokens = tokenizer.tokenize(words)

    weights = [idf_scores.get(token, 0.0) for token in tokens]
    
    return wv.get_mean_vector(tokens, weights = weights, pre_normalize = True, post_normalize = True)

In [466]:
from collections import Counter

def compute_idf(texts, wv, tokenizer):

    counts = Counter()
    
    for text in tqdm(texts, leave = False):

        words = text.lower()
        tokens = tokenizer.tokenize(words)
        tokens = [token for token in tokens if token in wv.key_to_index] 
        tokens = np.unique(tokens)
        counts.update(tokens)    
        
    idf = {token: np.log(len(texts)/max(count, 1)) for token, count in counts.items()}
    return idf

In [467]:
idf_scores = compute_idf(all_questions['text'], wv, nltk_tok)

  0%|          | 0/727692 [00:00<?, ?it/s]

In [485]:
all_embeddings = np.stack([compute_embed(text, wv, nltk_tok, idf_scores)for text in 
                            tqdm(all_questions['text'], leave = False)])

  0%|          | 0/727692 [00:00<?, ?it/s]

In [438]:
all_embeddings.shape

(727692, 300)

In [484]:
k = 10

for query in queries:
    query_vec = compute_embed(query, wv, nltk_tok, idf_scores)
    scores = all_embeddings.dot(query_vec)
    topk_idx = np.argsort(scores)[::-1][:k]

    print("query:",query)
    for i in topk_idx:
        print(f"{scores[i]:.3f}", all_questions['text'][i])
    print(100*'-')

query: How can I become healthier?
0.890 Is it healthier to be a vegetarian?
0.890 Is it healthier to be a vegetarian?
0.873 Is organic food safer and healthier?
0.864 How much healthier is it to be vegan than vegetarian?
0.848 How do I become stronger and smarter?
0.843 Do athletes live longer and healthier? If not, why?
0.842 Health: Why is it desirable to live a longer, healthier life?
0.838 Can India ever become healthy?
0.833 Do doctors really want people to live healthier?
0.833 Is organic food really healthier? Is it more nutritious?
----------------------------------------------------------------------------------------------------
query: How to reduce my sexual energy?
0.913 How do I decrease my sexual energy?
0.912 How do I reduce sexual energy?
0.876 What is the best way to convert chemical energy into electrical energy?
0.876 What is the best way to convert chemical energy into electrical energy?
0.876 What is the best way to convert chemical energy into electrical energy?
